# Aligning event streams to video time

This notebook shows how to use `video_alignment` to convert event times from
other data streams (spikes, fiber photometry, behavior events) into seconds
within the recorded video, so you can clip the video around those events.

The module is dependency-light (`pandas` only) and decoupled from the kinematics
pipeline. It works with three clocks:

| clock | zero point | used for |
|---|---|---|
| `behavior_time` | absolute harp / reference time | the video CSV `Behav_Time`, NWB `goCue_start_time`, spikes, FIP |
| `video_time` | first video frame = 0.0 | what `ffmpeg -ss` expects |
| `session_time` | first go cue = 0.0 | analyses aligned to the go cue |

Core relationships:

```
offset       = first_go_cue_time - first_frame_behavior_time   # video_time of the 1st go cue
video_time   = behavior_time     - first_frame_behavior_time   # behavior_time event -> video_time
video_time   = session_time      + offset                      # session_time event  -> video_time
```

In [1]:
import numpy as np
import pandas as pd

from aind_dynamic_foraging_behavior_video_analysis import video_alignment as va

## 1. A synthetic behavior video CSV

The real file is the Bonsai / AIND acquisition CSV (`metadata.csv`), written
**without a header** with column order
`['Behav_Time', 'Frame', 'Camera_Time', 'Gain', 'Exposure']`.

Here we fabricate a tiny one so the notebook runs anywhere: 500 Hz frames whose
first frame lands at `behavior_time = 100.0 s`.

In [2]:
n_frames = 5000
fps = 500.0
first_frame_behavior_time = 100.0  # harp seconds of the first frame

behav = first_frame_behavior_time + np.arange(n_frames) / fps
video_csv = pd.DataFrame({
    'Behav_Time': behav,
    'Frame': np.arange(n_frames),
    'Camera_Time': (np.arange(n_frames) * (1e9 / fps)).astype('int64'),  # ns
    'Gain': 1.0,
    'Exposure': 1.0,
})

video_csv_path = 'example_metadata.csv'
video_csv.to_csv(video_csv_path, header=False, index=False)  # headerless, like the real file
video_csv.head()

,Behav_Time,Frame,Camera_Time,Gain,Exposure
0,100.000,0,0,1.0,1.0
1,100.002,1,2000000,1.0,1.0
2,100.004,2,4000000,1.0,1.0
3,100.006,3,6000000,1.0,1.0
4,100.008,4,8000000,1.0,1.0


## 2. Read the first-frame time and compute the session↔video offset

`first_frame_behavior_time` is the only value read from the CSV. The `offset` is
that combined with the first go cue — it equals the `video_time` at which the
first go cue occurs.

In [3]:
# In real use, pull this from the NWB trials table, e.g.:
#   first_go_cue_time = float(nwb.trials['goCue_start_time'][0])
first_go_cue_time = 105.25

first_frame = va.get_first_frame_behavior_time(video_csv_path)
offset = va.compute_video_session_offset(video_csv_path, first_go_cue_time)

print(f'first_frame_behavior_time = {first_frame}')
print(f'offset (video_time of 1st go cue) = {offset}')

first_frame_behavior_time = 100.0
offset (video_time of 1st go cue) = 5.25


## 3. Convert event times to video_time

Pick the converter based on which clock your events are already on.

**Events on the raw behavior/harp clock** (e.g. spike or FIP times pulled straight
from NWB) — use `behavior_time_to_video_time`:

In [4]:
spike_times_behavior = np.array([102.0, 105.25, 108.5, 112.0])  # harp seconds
spike_times_video = va.behavior_time_to_video_time(spike_times_behavior, first_frame)
spike_times_video  # seconds into the video file

array([ 2.  ,  5.25,  8.5 , 12.  ])

**Events already zeroed to the first go cue** (`session_time`) — use
`session_time_to_video_time`. Both routes agree:

In [5]:
spike_times_session = spike_times_behavior - first_go_cue_time
via_session = va.session_time_to_video_time(spike_times_session, offset)

assert np.allclose(via_session, spike_times_video)
print('session-route and behavior-route agree:', via_session)

session-route and behavior-route agree: [ 2.    5.25  8.5  12.  ]


## 4. Clip the video around those events

Feed the `video_time` timestamps to the existing ffmpeg helper. This needs a real
video file and `ffmpeg` installed, so it is left as a commented sketch.

In [6]:
# from aind_dynamic_foraging_behavior_video_analysis.kinematics.video_clip_utils import (
#     extract_clips_ffmpeg_after_reencode,
# )
#
# clip_length = 2.0  # seconds
# pad = 0.5          # start each clip slightly before the event
# starts = spike_times_video - pad
# stems = [f'spike_{i}' for i in range(len(starts))]
#
# extract_clips_ffmpeg_after_reencode(
#     input_video_path='path/to/video.mp4',
#     timestamps=starts,
#     clip_length=clip_length + pad,
#     output_dir='clips_out',
#     filename_stems=stems,
# )

## Real-data recipe

```python
from aind_dynamic_foraging_behavior_video_analysis import video_alignment as va
from aind_dynamic_foraging_behavior_video_analysis.kinematics.tongue_kinematics_utils import (
    find_video_csv_path,
)

video_csv_path = find_video_csv_path(behavior_videos_path, camera_name='BottomCamera')
first_frame = va.get_first_frame_behavior_time(video_csv_path)

# any NWB-clock event stream -> video time
video_times = va.behavior_time_to_video_time(event_times_nwb, first_frame)
```